<a href="https://colab.research.google.com/github/Ramdharshan2007/DAA-Lab-Experiment/blob/main/10C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import random
import sys
import time

# Increase recursion limit to safely handle deep recursions if tested on larger inputs
sys.setrecursionlimit(30000)

class IntroSortComparison:
    def __init__(self):
        self.comparisons = 0

    # --- Heap Sort Components ---
    def heapify(self, arr, n, i, low):
        largest = i
        left = 2 * (i - low) + 1 + low
        right = 2 * (i - low) + 2 + low

        if left < low + n:
            self.comparisons += 1
            if arr[left] > arr[largest]:
                largest = left

        if right < low + n:
            self.comparisons += 1
            if arr[right] > arr[largest]:
                largest = right

        if largest != i:
            arr[i], arr[largest] = arr[largest], arr[i]
            self.heapify(arr, n, largest, low)

    def heap_sort(self, arr, low, high):
        n = high - low + 1
        # Build max heap
        for i in range(low + n // 2 - 1, low - 1, -1):
            self.heapify(arr, n, i, low)

        # Extract elements from heap one by one
        for i in range(high, low, -1):
            arr[low], arr[i] = arr[i], arr[low]
            self.heapify(arr, i - low, low, low)

    # --- Insertion Sort for small partitions ---
    def insertion_sort(self, arr, low, high):
        for i in range(low + 1, high + 1):
            key = arr[i]
            j = i - 1
            while j >= low:
                self.comparisons += 1
                if arr[j] > key:
                    arr[j + 1] = arr[j]
                    j -= 1
                else:
                    break
            arr[j + 1] = key

    # --- Partition Scheme (Median-of-Three Pivot) ---
    def partition(self, arr, low, high):
        mid = (low + high) // 2
        # Median of three logic to select a stable pivot
        if arr[low] > arr[mid]:
            arr[low], arr[mid] = arr[mid], arr[low]
        if arr[low] > arr[high]:
            arr[low], arr[high] = arr[high], arr[low]
        if arr[mid] > arr[high]:
            arr[mid], arr[high] = arr[high], arr[mid]

        # Place pivot at high - 1
        pivot_index = high - 1
        arr[mid], arr[pivot_index] = arr[pivot_index], arr[mid]
        pivot = arr[pivot_index]

        i = low
        j = high - 1
        while True:
            i += 1
            while True:
                self.comparisons += 1
                if arr[i] < pivot:
                    i += 1
                else:
                    break

            j -= 1
            while True:
                self.comparisons += 1
                if arr[j] > pivot:
                    j -= 1
                else:
                    break

            if i >= j:
                break
            arr[i], arr[j] = arr[j], arr[i]

        arr[i], arr[high - 1] = arr[high - 1], arr[i]
        return i

    # --- IntroSort Implementation ---
    def _introsort_util(self, arr, low, high, depth_limit):
        size = high - low + 1

        # Fallback to insertion sort for small sub-arrays
        if size < 16:
            self.insertion_sort(arr, low, high)
            return

        # Switch to Heap Sort if recursion depth exceeds 2 * log2(n)
        if depth_limit == 0:
            self.heap_sort(arr, low, high)
            return

        # Otherwise, proceed with Quick Sort partitioning
        p = self.partition(arr, low, high)
        self._introsort_util(arr, low, p - 1, depth_limit - 1)
        self._introsort_util(arr, p + 1, high, depth_limit - 1)

    def sort_introsort(self, arr):
        self.comparisons = 0
        n = len(arr)
        if n <= 1:
            return 0.0, 0

        # Calculate depth limit: 2 * floor(log2(n))
        depth_limit = 2 * math.floor(math.log2(n))

        start = time.perf_counter()
        self._introsort_util(arr, 0, n - 1, depth_limit)
        end = time.perf_counter()
        return end - start, self.comparisons

    # --- Standard Quick Sort (for comparison baseline) ---
    def _standard_quicksort(self, arr, low, high):
        if low < high:
            p = self.partition(arr, low, high)
            self._standard_quicksort(arr, low, p - 1)
            self._standard_quicksort(arr, p + 1, high)

    def sort_standard_quick(self, arr):
        self.comparisons = 0
        start = time.perf_counter()
        self._standard_quicksort(arr, 0, len(arr) - 1)
        end = time.perf_counter()
        return end - start, self.comparisons

def main():
    N = 10000
    print(f"--- IntroSort vs. Standard Quick Sort (Array Size N = {N}) ---")
    print(f"{'Configuration':<16} | {'Algorithm':<18} | {'Comparisons':<12} | {'Time (s)'}")
    print("-" * 65)

    base_rand = [random.randint(1, N) for _ in range(N)]
    configs = {
        "Random": base_rand,
        "Sorted": sorted(base_rand)
    }

    for config_name, arr in configs.items():
        # Test Standard Quick Sort
        qs = IntroSortComparison()
        try:
            t_std, c_std = qs.sort_standard_quick(arr.copy())
            std_time_str = f"{t_std:.6f}"
            std_comp_str = str(c_std)
        except RecursionError:
            std_time_str = "RECURSION ERROR"
            std_comp_str = "N/A"

        # Test IntroSort
        isort = IntroSortComparison()
        t_intro, c_intro = isort.sort_introsort(arr.copy())

        print(f"{config_name:<16} | {'Standard Quick':<18} | {std_comp_str:<12} | {std_time_str}")
        print(f"{'':<16} | {'IntroSort':<18} | {c_intro:<12} | {t_intro:.6f}")
        print("-" * 65)

if __name__ == "__main__":
    main()

--- IntroSort vs. Standard Quick Sort (Array Size N = 10000) ---
Configuration    | Algorithm          | Comparisons  | Time (s)
-----------------------------------------------------------------
Random           | Standard Quick     | 132199       | 0.018478
                 | IntroSort          | 139137       | 0.021207
-----------------------------------------------------------------
Sorted           | Standard Quick     | 140740       | 0.017869
                 | IntroSort          | 124876       | 0.013379
-----------------------------------------------------------------
